<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2026/images/logo.png?raw=true" alt="2026年度ゲノム情報解析入門" height="100px" align="middle">

<div align="right"><a href="https://github.com/CropEvol/lecture#section2">実習表ページに戻る</a></div>

リードアライメント、SNP検出
---

　前回、シーケンシングデータからゲノム配列の復元を試みました。できあがったゲノム配列は、他種のゲノム配列と比較したり、種内でのさまざまなゲノム解析のための基準配列（**参照配列、リファレンス配列、Reference sequence**）として利用できます。

　ただし、ゲノム配列の完全な復元にはそれなりにコストがかかります。そのため、複数のサンプル間でゲノム配列を比較する場合、全サンプルのゲノム配列を復元することは容易ではありません。そこで、サンプル間の比較を行う際には、そこまでコストをかけずにシーケンスデータを使用する方法が行われます。

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/mutation.png?raw=true" alt="title" height="150px">

　この実習テキストでは、次世代シーケンシングデータ（ショートリードデータ）を使って、どのようにサンプル間の変異（塩基の差異）を検出するかを勉強していきます。また、次回のQTL-seq解析法の基礎「バルクシーケンシング」を勉強していきます。

### サンプルデータのダウンロード

　この実習で使用するサンプルデータをダウンロードします。

- 参照配列:
  - 品種Aのゲノム配列（300bp）
- リード:
  - 品種Aのショートリード（各20bp）
  - 品種Bのショートリード（各20bp）
  - 品種Aと品種Bの交配F<small>1</small>世代1個体のショートリード（各20bp）
  - 品種Aと品種Bの交配F<small>2</small>世代100個体のショートリード（各20bp）


In [ ]:
%%bash
## サンプルファイルのダウンロード
wget -q -O 'alignment_dataset.sh' https://raw.githubusercontent.com/CropEvol/lecture/master/textbook_2025/scripts/alignment_dataset.sh
bash alignment_dataset.sh
## 表示
echo 'Downloaded dataset: genome/ reads/ bam/ ngs/'

### ソフトウェアのダウンロード

　今回使用するソフトウェアとその実行プログラムをダウンロードをします。

In [ ]:
%%bash
## ソフトウェアのインストール
wget -q -O 'alignment_softwares.sh' https://raw.githubusercontent.com/CropEvol/lecture/master/textbook_2025/scripts/alignment_softwares.sh
bash alignment_softwares.sh


# 今回の勉強内容

1. リードアライメントとSNP検出を学ぶ
  1. 品種Aのリードアライメント
  1. 品種BのリードアライメントとSNP検出
  1. 交配F<small>1</small>個体のアライメントデータ
  1. 交配F<small>2</small>個体のアライメントデータ
  1. 形質と連関しているSNP箇所を探す
  1. 全100個体で確かめる（遺伝子型の表をつくる）
  1. ジェノタイピングは本当に簡単か？
  1. シーケンシングの種類と特徴
1. バルクシーケンシングを学ぶ
  1. バルクシーケンシング
  1. 実例: イネのバルクシーケンシング解析

　今回は、Slido（スライド）を使ったアンケートが何回かあります。🗳 Slido の印があるところでは、先にSlidoで回答してから、次のコードセルを実行してください。


# 1. リードアライメントとSNP検出を学ぶ

例えば、イネは日本晴れという系統のゲノム配列が完全に復元されています。

そこで、コシヒカリと日本晴れのゲノム配列の違いを、コシヒカリのゲノム復元をせずに知りたい場合、下図の様な流れをとります。

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/short_read_mapping.png?raw=true" alt="title" height="500px">

コシヒカリをシーケンスし、ショートリードを読んで、アラインメントという過程を経て、配列の違いを確かめます。


## 1-1. 品種Aのリードアライメント

　最初に、 **リードアライメント（Read alignment）** について勉強していきましょう。リードアライメントは、基準となる配列（参照配列；通常は、同種の1個体のゲノム配列）に対して、シーケンシングデータ（**リード Reads**）を並べることを言います。その際、多少のミスマッチ（塩基の違い）は許容して、配列が似ている箇所にリードを並べます。

　リードアライメントをおこなうと、参照配列とリードの間の塩基の違いを容易に観察できるようになります。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/read_alignment.png?raw=true" alt="Read alignment" height="80px" align="middle">




　ここでは、以下2つの配列データを使ってリードのアラインメントを行ってみます。
- 参照配列: 品種Aのゲノム配列（300bp）
  - /content/genome/CultivarA.fa
- リード: 品種Aのリード（合計約240本の**ペアエンドリード**<sup>※2</sup>、各20bp）
  - /content/reads/CultivarA_read1.fq
  - /content/reads/CultivarA_read2.fq

**品種Aのゲノム配列**に**品種Aをシーケンス**して得られたリードをアラインメントします。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/paired_end_read.png?raw=true" alt="DNA sequencing and paired-end read" height="230px" align="middle">

　リードのアラインメントはソフトウェアを使用することで簡単に行う事が出来ます。

　実際のゲノム解析でも良く使われている[**BWA (Burrows-Wheeler Alignment)**](https://bio-bwa.sourceforge.net/)と呼ばれるソフトウェアで、リードアライメントをおこなってみましょう。

　次のプログラムを実行して、リードアライメントをおこなってください。

<small>※1 リードアライメントの仕組みに興味がある方は次のWebページを参照してください。[How Do We Compare Biological Sequences?](https://www.bioinformaticsalgorithms.org/bioinformatics-chapter-5) (Bioinformatics Algorithms / Phillip Compeau)</small>


In [ ]:
%%bash
## リードアライメント: 品種Aのリードを、品種Aのゲノム配列にアライメントする

## リードアライメントのための事前準備（ゲノム配列のインデックス作成）
if [ ! -e genome/CultivarA.fa.bwt ]; then bwa index genome/CultivarA.fa; fi

## リードアライメント
bwa aln genome/CultivarA.fa reads/CultivarA_read1.fq > read1.sai 2>/dev/null
bwa aln genome/CultivarA.fa reads/CultivarA_read2.fq > read2.sai 2>/dev/null
bwa sampe genome/CultivarA.fa read1.sai read2.sai reads/CultivarA_read1.fq reads/CultivarA_read2.fq > CultivarA.sam
samtools sort -O bam CultivarA.sam > CultivarA.bam
samtools index CultivarA.bam
rm -fR read1.sai read2.sai

プログラムが走り終えると、`CultivarA.sam`や`CultivarA.bam`というファイルが出来ているかと思います。

#### **SAMファイル/BAMファイル**

**SAMファイル**は**Sequence Alignment and Map**ファイルのことを指し、**BAMファイル**は**Binary Alignment and Map**ファイルというSAMファイルのデータ量を減らしたファイルになります。

今回実施したショートリードのアラインメント結果の情報が保存されています。

どのリードが、どの場所にアラインメントされ、そのクオリティがどの程度なのか…等の情報を見る事が出来ます。

In [ ]:
%%bash
# SAMファイルの中身を一部確認
head -n 10 CultivarA.sam

　アライメント結果は、[**IGV (Integrative Genomics Viewer)**](https://software.broadinstitute.org/software/igv/home)で確認する事が出来ます。

IGVはこれまでにも何度か使用してきましたが、主にこの様なアラインメントの結果を見るために使用されることが多い可視化ツールになります。

In [ ]:
## アライメント結果を見る
## 準備: ライブラリ
import igv_notebook
from igv_prep import RefTrack, AnnotationTrack, BamTrack
igv_notebook.init()
## 準備: Reference info
ref = RefTrack({ "fastaPath":"genome/CultivarA.fa", "indexPath":"genome/CultivarA.fa.fai", "id":"CultivarA" })
## 準備: Reads info
A = BamTrack({ "name":"CultivarA", "path":"CultivarA.bam", "indexPath":"CultivarA.bam.bai", "viewAsPairs":True })
## IGV表示
b = igv_notebook.Browser(ref)
b.load_track(A)

　一部のリード内には、参照配列と異なる塩基が見つかってきます。その異なる塩基には二つの可能性があります。
- シーケンシングしたサンプル内にみられる変異（**一塩基多型; Single nucleotide polymorphism, SNP**）
- シーケンシング中に発生する塩基の読み間違い（シーケンシングエラー; Sequencing error）

　じゅうぶんな数のリードがあれば、その異なる塩基がSNPかシーケンシングエラーかを判別できます。

　今回の品種Aのリードでは、「参照配列と異なる塩基」は大部分のリードが共通にもっているわけではありません。そのため、見つかった「参照配列と異なる塩基」は、シーケンシングエラーと考えるのがよさそうです。

## 1-2. 品種BのリードアライメントとSNP検出

　今度は、品種Bのリードを品種Aの参照配列に対してアライメントしてみましょう。

使用するデータ:
- 参照配列: 品種Aのゲノム配列（300bp）
  - /content/genome/CultivarA.fa
- リード配列: 品種Bのリード配列（合計約240本のペアエンドリード、各20bp）
  - /content/reads/CultivarB_read1.fq
  - /content/reads/CultivarB_read2.fq

**品種Aのゲノム配列**に**異なる品種(品種B)をシーケンス**して得られたリードをアラインメントします。


In [ ]:
%%bash
## リードアライメント: 品種Bのリードを、品種Aのゲノム配列にアライメントする

## リードアライメントの事前準備
if [ ! -e genome/CultivarA.fa.bwt ]; then bwa index genome/CultivarA.fa; fi
## リードアライメント
bwa aln genome/CultivarA.fa reads/CultivarB_read1.fq > read1.sai 2>/dev/null
bwa aln genome/CultivarA.fa reads/CultivarB_read2.fq > read2.sai 2>/dev/null
bwa sampe genome/CultivarA.fa read1.sai read2.sai reads/CultivarB_read1.fq reads/CultivarB_read2.fq > CultivarB.sam
samtools sort -O bam CultivarB.sam > CultivarB.bam
samtools index CultivarB.bam
rm -fR read1.sai read2.sai

In [ ]:
## アライメント結果を見る
## 準備: ライブラリ
import igv_notebook
from igv_prep import RefTrack, AnnotationTrack, BamTrack
igv_notebook.init()
## 準備: Reference info
ref = RefTrack({ "fastaPath":"genome/CultivarA.fa", "indexPath":"genome/CultivarA.fa.fai", "id":"CultivarA" })
## 準備: Reads info
B = BamTrack({ "name":"CultivarB", "path":"CultivarB.bam", "indexPath":"CultivarB.bam.bai", "viewAsPairs":True })
## IGV表示
b = igv_notebook.Browser(ref)
b.load_track(B)

　アライメントの結果、一部の塩基箇所において、ほぼすべてのリードが異なる塩基を共通にもっています。その塩基の違いは、シーケンシングエラーではなく、品種Aと品種Bの間の**SNP**変異と考えられます。

今回のケースだと、30,	41,	55,	88,	120,	134,	176,	198,	213,	225,	270, 285 bpの12箇所に品種AとBの間の変異があります。

## 1-3. 交配F<small>1</small>世代のアライメントデータ

　品種Aと品種Bを人為的に交配して得られた個体（F<small>1</small>個体）では、どのようなアライメント結果が得られるか見てみましょう。以下を実行すると、F<small>1</small>個体のリードのアライメント結果を表示できます。

<small>※ ここからは、bwaによるリードアライメントを省略し、事前に用意したアライメント結果を見ていきます。</small>

> ### 🗳 Slido Q1
> **品種A（SNP箇所の塩基がA型）と品種B（B型）を交配したF<small>1</small>個体のリードを、品種Aの参照配列にアライメントしました。SNP箇所のリードの塩基は、どのように見えるでしょうか？**
>
> - ほぼすべてのリードが品種Aと同じ塩基
> - ほぼすべてのリードが品種Bと同じ塩基
> - 品種Aの塩基と品種Bの塩基が、およそ半分ずつ
> - リードごとにばらばら（A・C・G・Tが混ざる）
>
> <small>下のコードセルを実行する前に回答してください。</small>

In [ ]:
## アライメント結果を見る
## 準備: ライブラリ
import igv_notebook
from igv_prep import RefTrack, AnnotationTrack, BamTrack
igv_notebook.init()
## 準備: Reference info
ref = RefTrack({ "fastaPath":"genome/CultivarA.fa", "indexPath":"genome/CultivarA.fa.fai", "id":"CultivarA" })
## 準備: Reads info
B = BamTrack({ "name":"CultivarB", "path":"bam/CultivarB.bam", "indexPath":"bam/CultivarB.bam.bai", "viewAsPairs":True, "displayMode":"SQUISHED", "height":150 })
F1= BamTrack({ "name":"F1",       "path":"bam/F1.bam",       "indexPath":"bam/F1.bam.bai",       "viewAsPairs":True })
## IGV表示
b = igv_notebook.Browser(ref)
b.load_track(B)
b.load_track(F1)

　F<small>1</small>個体は、品種Aと品種Bから一つずつ配偶子を受け取って生じた個体であり、それぞれの品種に由来するSNP塩基（以下、**SNPアリル、SNP allele**）をもっているはずです。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/f1_hybrid.png?raw=true" alt="SNP allele heterozygous of F1" height="200px" align="middle">

　アライメント結果の各SNP箇所の塩基構成をみると、多少ばらついているものの、二つのSNPアリルがおよそ50%の割合で存在しています。

　このように、どちらかのSNPアリルに固定していない状態を、 **ヘテロ接合型（Heterozygote）** と言います。一方で、品種Aや品種Bにみられるように、どちらかのSNPアリルに固定された状態を、 **ホモ接合型（Homozygote）** と言います。SNPアリルの保有状況から遺伝子型（ヘテロ接合型かホモ接合型か）を決めることを **ジェノタイピング（Genotyping）** といいます。




## 1-4. 交配F<small>2</small>世代のアライメントデータ

　F<small>1</small>個体を自殖し、F<small>2</small>世代の子孫を得ると、SNPアリルは各個体にランダムに遺伝します。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/SNP_allele_segregation.png?raw=true" alt="SNP allele segregation in F2" height="230px" align="middle">

　F<small>2</small>世代では各個体のリードアライメントがどのようになるか見てみましょう。次のプログラムを実行すると、F<small>2</small>世代の100個体のうち、指定のF<small>2</small>個体（例えば、F2_001からF2_005）のアライメント結果を見れます。

> ### 🗳 Slido Q2
> **F<small>1</small>個体を自殖して得たF<small>2</small>世代100個体では、あるSNP箇所の遺伝子型（AA: 品種Aのホモ接合、AB: ヘテロ接合、BB: 品種Bのホモ接合）は、およそ何個体ずつになると予想しますか？**
>
> <small>AA・AB・BBの個体数を「25, 50, 25」のように数字で回答してください。答えは「1-6. 全100個体で確かめる」で確認します。</small>

In [ ]:
## 表示するF2番号: 1~100までの数字を自由に組み合わせ可能
## 注意： 数字と数字の間には半角のカンマ(,)が必要です。
F2 = [1,2,3,4,5]

## 準備: ライブラリ
import igv_notebook
from igv_prep import RefTrack, AnnotationTrack, BamTrack
bam_common_info = { "displayMode":"SQUISHED", "viewAsPairs":False, "height":100, "showCoverage":False }
igv_notebook.init()
## 準備: Reference info
ref = RefTrack({ "fastaPath":"genome/CultivarA.fa", "indexPath":"genome/CultivarA.fa.fai", "id":"CultivarA" })
## 準備: Reads info
Info = [BamTrack({ **{"name":"CultivarB", "path":"bam/CultivarB.bam", "indexPath":"bam/CultivarB.bam.bai"}, **bam_common_info }),
        BamTrack({ **{"name":"F1",       "path":"bam/F1.bam", "indexPath":"bam/F1.bam.bai"}, **bam_common_info })]
for f in sorted(list(set(F2))):
    Info += [ BamTrack({ **{"name":"F2_{:03}".format(f),
                          "path":"/content/bam/F2_{:03}.bam".format(f),
                          "indexPath":"/content/bam/F2_{:03}.bam.bai".format(f)}, **bam_common_info }) ]
## IGV表示
b = igv_notebook.Browser(ref)
for i in Info: b.load_track(i)

　F<small>2</small>世代の個体ごとに各SNP箇所のSNPアリルの構成が異なっており、SNPアリルの分離を確認できました。

　F<small>2</small>個体集団のように、SNPアリルがその場所ごとに独立に分離した（染色体がシャッフルされた）集団は、両親間の形質の違いを生み出す遺伝子（原因遺伝子）のマッピングに役立ちます。

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/chromosome_shuffling.png?raw=true" alt="Chromosome shuffling" height="230px" align="middle">







## 1-5. 形質と連関しているSNP箇所を探す
　つぎは、形質情報も考慮し、リードアライメント結果を見ていきましょう。各個体の形質データは次のとおりです。
- F<small>2</small>各個体の形質: [形質データ表](https://github.com/CropEvol/lecture/blob/master/textbook_2025/dataset/f2_phenotypes.pdf)
- 品種Aの形質: Short
- 品種Bの形質: Long
- F<small>1</small>個体の形質: Medium


　品種Aと同じ「Short」形質をもつF<small>2</small>個体を4つ程度選び、それらのアライメント結果を表示してください。さらに、そのグループ内で分離していないSNP（固定SNP）を探してみてください。


In [ ]:
## 品種Aと同じShort形質をもつF2個体番号を半角数字で入力してください。
## 注意： 数字と数字の間には半角のカンマ(,)が必要です。
F2_S_bulk = [] # F2_001, F2_002, F2_003, F2_004を選びたければ[1, 2, 3, 4]とする。

## 準備: ライブラリ
import igv_notebook
from igv_prep import RefTrack, AnnotationTrack, BamTrack
bam_common_info = { "displayMode":"SQUISHED", "viewAsPairs":False, "height":100, "showCoverage":False }
igv_notebook.init()
## 準備: Reference info
ref = RefTrack({ "fastaPath":"genome/CultivarA.fa", "indexPath":"genome/CultivarA.fa.fai", "id":"CultivarA" })
## 準備: Reads info
Info = [BamTrack({ **{"name":"CultivarA", "path":"bam/CultivarA.bam", "indexPath":"bam/CultivarA.bam.bai"}, **bam_common_info }),
        BamTrack({ **{"name":"CultivarB", "path":"bam/CultivarB.bam", "indexPath":"bam/CultivarB.bam.bai"}, **bam_common_info }),]
for f in sorted(list(set(F2_S_bulk))):
    Info += [ BamTrack({ **{"name":"F2_{:03}".format(f),
                          "path":"bam/F2_{:03}.bam".format(f),
                          "indexPath":"bam/F2_{:03}.bam.bai".format(f)}, **bam_common_info }) ]
## IGV表示
b = igv_notebook.Browser(ref)
for i in Info: b.load_track(i)

　同様に、品種Bと同じ「Long」形質のF<small>2</small>個体についても、4個体分程度のアライメント結果を表示し、固定SNPを探してください。

In [ ]:
## 品種Bと同じLong形質をもつF2個体番号を半角数字で入力してください。
## 注意： 数字と数字の間には半角のカンマ(,)が必要です。
F2_L_bulk = []

## 準備: ライブラリ
import igv_notebook
from igv_prep import RefTrack, AnnotationTrack, BamTrack
bam_common_info = { "displayMode":"SQUISHED", "viewAsPairs":False, "height":100, "showCoverage":False }
igv_notebook.init()
## 準備: Reference info
ref = RefTrack({ "fastaPath":"genome/CultivarA.fa", "indexPath":"genome/CultivarA.fa.fai", "id":"CultivarA" })
## 準備: Reads info
Info = [BamTrack({ **{"name":"CultivarA", "path":"bam/CultivarA.bam", "indexPath":"bam/CultivarA.bam.bai"}, **bam_common_info }),
        BamTrack({ **{"name":"CultivarB", "path":"bam/CultivarB.bam", "indexPath":"bam/CultivarB.bam.bai"}, **bam_common_info }),]
for f in sorted(list(set(F2_L_bulk))):
    Info += [ BamTrack({ **{"name":"F2_{:03}".format(f),
                          "path":"bam/F2_{:03}.bam".format(f),
                          "indexPath":"bam/F2_{:03}.bam.bai".format(f)}, **bam_common_info }) ]
## IGV表示
b = igv_notebook.Browser(ref)
for i in Info: b.load_track(i)

　形質ごとに4個体程度を表示し、各SNP箇所のSNPアリル構成を見ていくと、前から55番目の塩基に固定SNPを発見できます。

55番目の塩基:
- Short形質の品種Aでは「A」
- Short形質のF<small>2</small>個体でも「A」のみ（品種A由来のアリルのホモ接合）
- Long形質の品種Bでは「T」
- Long形質のF<small>2</small>個体でも「T」のみ（品種B由来のアリルのホモ接合）

　形質とSNPアリルに連関（Association）がみられることから、形質の違いを作っている原因変異かもしれません。

<small>※ 「Medium」形質のF<small>2</small>個体のアライメント結果には、55番塩基に2つのSNPアリルが見つかるはずです。余裕があれば確認してみてください。</small>


## 1-6. 全100個体で確かめる（遺伝子型の表をつくる）

　1-5 では、IGVで4個体ずつ見比べて、55番目の塩基に形質と連関するSNPを見つけました。では、F<small>2</small>の**全100個体**でも同じことが言えるでしょうか？ 100個体をIGVで1つずつ見るのは大変なので、プログラムで調べてみましょう。

　次のコードセルでは、次の3つの作業をまとめておこないます（中身を理解する必要はありません）。

1. 品種Bのアライメント結果から、品種Aと品種Bの間のSNPの位置を探す
1. F<small>2</small>の全100個体について、各SNP位置の「品種A型の塩基（Aアリル）」と「品種B型の塩基（Bアリル）」のリードの本数を数える
1. 各個体の遺伝子型を決める（ジェノタイピング）
   - 品種A型のリードしかない → AA（品種A型のホモ接合）
   - 品種B型のリードしかない → BB（品種B型のホモ接合）
   - 両方のリードがアラインメントされる → AB（ヘテロ接合）

In [ ]:
import re, subprocess
import pandas as pd

## F2各個体の形質（形質データ表の内容。S=Short, M=Medium, L=Long。左から F2_001, F2_002, ...）
pheno_code = ("MMMLSMSLMS" "LMMMSMMSSS" "LMSMMLSSML" "MLLMLMSMMS" "MSLMMMLMSM"
              "SMLSMMLSLS" "SMSLMLMMLM" "LMMSMSMLMM" "SSLMMSLMLL" "LSSMSMMSSM")
names = ["F2_{:03}".format(i) for i in range(1, 101)]
phenotype = pd.Series([{"S": "Short", "M": "Medium", "L": "Long"}[c] for c in pheno_code], index=names)

def count_bases(bams, region=None, positions=None):
    ## samtools mpileup で、各位置・各サンプルの「参照配列と同じ塩基」「違う塩基」の本数を数える
    cmd = ["samtools", "mpileup", "-A", "-B", "-Q", "0", "-x", "-d", "0", "-f", "genome/CultivarA.fa"]
    if region:    cmd += ["-r", region]
    if positions: cmd += ["-l", positions]
    out = subprocess.run(cmd + bams, capture_output=True, text=True).stdout
    rows = []
    for line in out.splitlines():
        c = line.split("\t")
        pos, ref = int(c[1]), c[2].upper()
        for k, bam in enumerate(bams):
            s = c[4 + 3*k]
            s = re.sub(r"\^.", "", s).replace("$", "")              # リードの始まり・終わりの記号を除く
            while True:                                              # 挿入・欠失（+2AC など）を除く
                m = re.search(r"[+-](\d+)", s)
                if not m: break
                s = s[:m.start()] + s[m.end() + int(m.group(1)):]
            bases = [ref if b in ".," else b.upper() for b in s if b in ".,ACGTacgt"]
            for b in "ACGT":
                rows.append((bam, pos, ref, b, bases.count(b)))
    return pd.DataFrame(rows, columns=["bam", "pos", "ref", "base", "count"])

## (1) 品種Bのリードから、SNPの位置と品種Bの塩基を求める
b = count_bases(["bam/CultivarB.bam"], region="chr1")
b = b.pivot_table(index=["pos", "ref"], columns="base", values="count").reset_index()
b["depth"] = b[list("ACGT")].sum(axis=1)
b["major"] = b[list("ACGT")].idxmax(axis=1)
snps = b[(b["major"] != b["ref"]) & (b[list("ACGT")].max(axis=1) / b["depth"] > 0.5)]
snps = snps.set_index("pos")[["ref", "major"]].rename(columns={"ref": "A_allele", "major": "B_allele"})
print("品種A・B間のSNP:", len(snps), "か所")
print(snps.T)

## (2) F2全100個体について、SNP位置のAアリル・Bアリルの本数を数える
with open("snp_positions.txt", "w") as f:
    for pos in snps.index:
        f.write(f"chr1\t{pos}\n")
c = count_bases([f"bam/{n}.bam" for n in names], positions="snp_positions.txt")
c["sample"] = c["bam"].str.extract(r"(F2_\d+)")[0]
c = c.pivot_table(index=["sample", "pos"], columns="base", values="count").reset_index()
c["A_count"] = [int(r[snps.loc[r["pos"], "A_allele"]]) for _, r in c.iterrows()]
c["B_count"] = [int(r[snps.loc[r["pos"], "B_allele"]]) for _, r in c.iterrows()]
allele_counts = c[["sample", "pos", "A_count", "B_count"]]

## (3) Bアリルの割合から遺伝子型を決める: 0.2未満→AA, 0.8超→BB, それ以外→AB
def call(a, b):
    if a + b == 0: return "?"
    r = b / (a + b)
    return "AA" if r == 0 else ("BB" if r == 1 else "AB")
allele_counts = allele_counts.assign(genotype=[call(a, b) for a, b in zip(allele_counts["A_count"], allele_counts["B_count"])])
genotypes = allele_counts.pivot(index="sample", columns="pos", values="genotype")
genotypes.insert(0, "phenotype", phenotype)
genotypes

　表の行が個体、列がSNPの位置（何番目の塩基か）です。表のままでは見づらいので、色を付けて、形質ごとに並べ替えてみましょう。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

## 形質の順（Short → Medium → Long）に並べ替える
order = {"Short": 0, "Medium": 1, "Long": 2}
g = genotypes.sort_values("phenotype", key=lambda s: s.map(order), kind="stable")
values = g.drop(columns="phenotype").replace({"AA": 0, "AB": 1, "BB": 2, "?": float("nan")}).astype(float)

colors = ["#4C78A8", "#E0E0E0", "#F58518"]   # AA: 青, AB: 灰, BB: 橙
fig, ax = plt.subplots(figsize=(8, 10))
ax.imshow(values, aspect="auto", cmap=ListedColormap(colors), interpolation="none")
ax.set_xticks(range(values.shape[1])); ax.set_xticklabels(values.columns)
ax.set_xlabel("SNP position (bp)")
## 形質グループの境目に線を引き、グループ名を書く
counts = g["phenotype"].value_counts()
y = 0
for ph in ["Short", "Medium", "Long"]:
    ax.text(-0.8, y + counts[ph] / 2, f"{ph}\n(n={counts[ph]})", ha="right", va="center")
    y += counts[ph]
    ax.axhline(y - 0.5, color="black", lw=1.5)
ax.set_yticks([])
ax.legend(handles=[Patch(color=c, label=l) for c, l in zip(colors, ["AA", "AB", "BB"])],
          loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

#### 図の見方
- 横軸: SNPの位置（参照配列の何番目の塩基か）
- 縦軸: F<small>2</small>の個体（上から Short・Medium・Long の順に並べています）
- 色: 遺伝子型（青: AA、灰: AB、橙: BB）

形質のグループと遺伝子型の色がきれいにそろっているのは、どのSNPの列かが目で見て分かるようになりました。

この情報は形質に影響を与えているSNPの場所を表しています。

　形質と遺伝子型の関係を、数でも確かめてみましょう。次のコードセルの `snp` を 55 → 41 → 120 と変えて実行してください。

In [ ]:
snp = 55   # 調べるSNPの位置（30, 41, 55, 88, 120, 134, 176, 198, 213, 225, 270, 285）

table = pd.crosstab(genotypes["phenotype"], genotypes[snp]).reindex(["Short", "Medium", "Long"])
print(f"{snp}番目のSNP: 形質ごとの遺伝子型の個体数")
table

　55番目のSNPでは、Short形質の個体はすべてAA、Long形質の個体はすべてBBでした。55番目から離れたSNPほど、形質と遺伝子型の対応が崩れていきます。これは、離れた場所ほど、その間で染色体の組換えが起きる機会が多いためです。

　原因遺伝子の近くのSNPほど形質との連関が強く、離れるほど弱くなる。この性質を使って、原因遺伝子の場所を探すのが**遺伝子マッピング**です。次回のQTL-seq解析では、このパターンを数万個のSNPについて調べます。

#### Slido Q2 の答え合わせ
　最後に、F<small>2</small>集団での遺伝子型の比を確かめましょう。メンデルの法則から予想される比は AA : AB : BB = 1 : 2 : 1 です。55番目のSNPの遺伝子型と、形質の個体数を数え、カイ二乗検定で 1 : 2 : 1 と矛盾しないか調べます。

In [ ]:
from scipy.stats import chisquare

observed_g = genotypes[55].value_counts().reindex(["AA", "AB", "BB"]).fillna(0)
observed_p = phenotype.value_counts().reindex(["Short", "Medium", "Long"])
for label, obs in [("55番目の遺伝子型 (AA, AB, BB)", observed_g), ("形質 (Short, Medium, Long)", observed_p)]:
    expected = [float(obs.sum() * r) for r in (0.25, 0.5, 0.25)]
    p = chisquare(obs, expected).pvalue
    print(f"{label}: 観察値 {list(obs.astype(int))}, 期待値 {expected}, p値 = {p:.3f}")

　p値が0.05より小さければ、観察された比は 1 : 2 : 1 と矛盾していると考えられます。逆に0.05より大きければ、1 : 2 : 1として許容できる範囲です。

## 1-7. ジェノタイピングは本当に簡単か？

　リードのアラインメント結果を詳しく見ると、例えばF2_017の個体では55番目のSNPをヘテロ(AB)と判定しています。

品種A型のリードが9本、B型のリードが2本アラインメントされていたためです。

In [ ]:
## F2_017の55番目のSNP
allele_counts[(allele_counts["pos"] == 55) & (allele_counts["sample"].isin(["F2_017"]))]

　このとき、Aアリル 9本・Bアリル 2本で、ヘテロ(AB)と判定されましたが、このBアリルの2本は、次のどちらなのでしょうか。

- 本当にヘテロ接合（AB）で、たまたまBアリルのリードが少なく読まれた
- 本当はホモ接合（AA）で、Bアリルの2本はシーケンシングエラー

> ### 🗳 Slido Q3
> **F2_017 の55番目の遺伝子型は、実際にはヘテロ接合（AB）とホモ接合（AA）のどちらだと思いますか？**
>
> - ヘテロ接合（AB）
> - ホモ接合（AA）
>
> <small>直感で構いません。次のコードセルで確率を計算して確かめます。</small>

　それぞれの場合に、「11本中Bアリルが2本」という結果がどのくらいの確率で起こるか、計算してみましょう。

- **ヘテロ接合の場合**: 各リードは50%の確率でBアリルになります。コインを11回投げて、表が2回以下になる確率と同じです。
- **ホモ接合（AA）の場合**: Bアリルのリードは、シーケンシングエラーでしか生じません。エラー率は、品種Aのリードを品種Aの参照配列に並べたときに、参照配列と違う塩基が読まれた割合から推定します。

In [ ]:
from scipy.stats import binom

n, k = 11, 2   # リードの本数, Bアリルの本数
## ヘテロ接合: Bアリルが k 本以下になる確率
p_het = binom.cdf(k, n, 0.5)
## ホモ接合（AA）: エラーでBアリルと同じ塩基が k 本以上読まれる確率
## （エラーで生じる塩基はA以外の3種類のどれかなので、Bアリルの塩基になる確率は エラー率 / 3）
p_hom = binom.sf(k - 1, n, error_rate / 3)
print(f"ヘテロ接合で「{n}本中Bアリルが{k}本以下」になる確率: {p_het:.4f}")
print(f"ホモ接合で「{n}本中Bアリルが{k}本以上」になる確率  : {p_hom:.4f}")
print(f"→ ヘテロ接合のほうが約 {p_het / p_hom:.0f} 倍起こりやすい")

　計算の結果、F2_017 はヘテロ接合である可能性のほうがずっと高いことがわかりました。表現型（Medium）とも一致します。

　とはいえミスの可能性は0では無いので、リードの本数が少ないと判定を誤ることがあります。リードの本数（**リードの深さ、Depth**）が多いほど、偶然の偏りは小さくなり、判定は正確になります。実際のジェノタイピングソフトウェア（bcftools や GATK など）は、このような確率の計算をもとに遺伝子型を決めています。

### 練習
　下のコードセルの `n` と `k` を変えて、リードの深さが増えるとどうなるか調べてみましょう。

たとえば、リード数が更に増えた場合として、`n, k = 22, 2` や `n, k = 44, 2` を試してください。

100％完全に正しいシーケンスというのはまだ難しいので、この様な誤差が生じることは頭において解析などを進める必要があります。

In [ ]:
from scipy.stats import binom

n, k = 11, 2   # リードの本数, Bアリルの本数
## ヘテロ接合: Bアリルが k 本以下になる確率
p_het = binom.cdf(k, n, 0.5)
## ホモ接合（AA）: エラーでBアリルと同じ塩基が k 本以上読まれる確率
## （エラーで生じる塩基はA以外の3種類のどれかなので、Bアリルの塩基になる確率は エラー率 / 3）
p_hom = binom.sf(k - 1, n, error_rate / 3)
print(f"ヘテロ接合で「{n}本中Bアリルが{k}本以下」になる確率: {p_het:.4f}")
print(f"ホモ接合で「{n}本中Bアリルが{k}本以上」になる確率  : {p_hom:.4f}")
print(f"→ ヘテロ接合のほうが約 {p_het / p_hom:.0f} 倍起こりやすい")

## 1-8. シーケンシングの種類と特徴

　各個体の遺伝子型決定（ジェノタイピング）によく使われるシーケンシング方法について触れておきます。

　先ほどリードの本数が多いほど正確に遺伝子型を決められる(ジェノタイピング)という話をしました。しかし、冒頭でも述べた通り、シーケンスには費用が掛かります。そのため、出来るだけ低コストで遺伝子型を決める(Genotyping)を行う方法が開発されています。

　単純に全ゲノム領域をシーケンスする全ゲノムシーケンスだけではなく、遺伝子の転写物のみをシーケンスするRNAシーケンス(後半の講義で扱います)、また一部の領域のみシーケンスするRADシーケンスなどがあります。

- 全ゲノムシーケンシング（Whole genome sequencing: WGS）
- RNAシーケンシング（RNA sequencing: RNA-seq）
- RADシーケンシング（Restriction site associated DNA (RAD) sequencing: RAD-seq）


In [ ]:
## 準備: ライブラリなど
import igv_notebook
from igv_prep import RefTrack, AnnotationTrack, BamTrack
igv_notebook.init()
base = "/content/ngs/"
bam_common_info = { "showCoverage":False, "displayMode":"SQUISHED", "viewAsPairs":True, "height":100 }

## 準備: Reference info
ref = RefTrack({ "fastaPath":base+"Ref.genome.fa", "indexPath":base+"Ref.genome.fa.fai", "id":"Reference" }, locus="chr01:10,000-35,000")
genes = AnnotationTrack({ "name":"Genes", "path":base+"Ref.transcript.gff", "format":"gff", "displayMode":"SQUISHED" })
restsite= AnnotationTrack({ "name":"Restriction site", "path":base+"Ref.radseq_enzymes.bed", "format":"bed", "color":"#ff7f0e"})
## 準備: Reads info
Info = [BamTrack({ **{"name":"WGS", "path":base+"wgs.bam", "indexPath":base+"wgs.bam.bai"}, **bam_common_info }),
        BamTrack({ **{"name":"RNA-seq", "path":base+"rnaseq.bam", "indexPath":base+"rnaseq.bam.bai"}, **bam_common_info }),
        BamTrack({ **{"name":"RAD-seq", "path":base+"radseq.bam", "indexPath":base+"radseq.bam.bai"}, **bam_common_info }),]
## IGV表示
b = igv_notebook.Browser(ref)
for i in Info: b.load_track(i)
b.load_track(genes)
b.load_track(restsite)

　それぞれのシーケンシング対象と得られるSNP、特徴をまとめると次のとおりです。

ゲノムサイズが大きい生物を扱う場合は、RADシーケンスなどの手法を使用することが多いです。限られた領域に絞って読むことで、同じデータ量でもその領域だけはリードをたくさん読む事が出来るという戦略です。

| 種類 | シーケンシング対象領域 / SNP探索領域 | 特徴 |
|:---|:---|:---|
| Whole genome sequencing | ゲノム全配列 　| 解析には多くのリードを必要とする。 |
| RNA sequencing | 発現遺伝子（mRNA）配列<br>（イントロンは含まない）　| 得られるリード量は遺伝子発現量に依存する。<br>必要なリード量もWGSに比べて少なく済む。 |
| RAD sequencing | 任意の制限酵素切断部位近くの配列　| 少ないリード量で解析をおこなえる。<br>類似シーケンシング手法: GBS, GRAS-Di, MIG-seq |


# 2. バルクシーケンシングを学ぶ

## 2-1. バルクシーケンシング

　ここまでは、各個体のアライメント結果をサンプルごとに表示しました。次は、**バルクシーケンシング** について簡単に触れておきます。

バルクシーケンシングとは、いくつかの個体のDNAをランダムに集めてプーリングし、"1サンプル分"のデータとして扱ったものです。

<img src="https://github.com/slt666666/FAO_lecture/blob/main/FAO_2024/Mutmap_8.png?raw=true" alt="title" height="200px">

　次のコードセルの`F2_S_bulk`（Short形質グループ: S-bulk）と`F2_L_bulk`（Long形質グループ: L-bulk）の右辺の括弧`[]`内に、4個体程度の個体番号を入力して、バルクシーケンシングリードのアライメント結果を見てみましょう。

- F<small>2</small>各個体の形質: [形質データ表](https://github.com/CropEvol/lecture/blob/master/textbook_2025/dataset/f2_phenotypes.pdf)

In [ ]:
## プーリングするF2個体の番号
## 注意： 数字と数字の間には半角のカンマ(,)が必要です。
F2_S_bulk = []
F2_L_bulk = []

## 以下、変更不要
## Pooling
import re
def Pooling(Ns, bam_dir, samplingRate=-1, output="bulk.bam", verbose=False, samtools_path="samtools"):
    f2_list = [bam_dir+"/F2_{:03}.bam".format(f) for f in sorted(list(set(Ns)))]
    if samplingRate>=1.0:
        samplingRate = 1.0
    elif samplingRate<=0.0:
        samplingRate = float(2.0 / len(Ns))  # 個体数に応じて、およそ2個体分のリード量にする
    f2_str = " ".join(f2_list)
    # リードのサンプリング
    !$samtools_path merge -o - $f2_str | samtools view -b -s $samplingRate > $output
    !$samtools_path index $output
    if verbose:
      f2_samples = [re.search(r".*/(.+?)$", i).group(1) for i in f2_list]
      print("Output: ", output)
      print("- Reads sampled from: "+",".join(f2_samples))
      print("- Sampling rate: %.3f" % samplingRate)

Pooling(F2_S_bulk, bam_dir="bam/", output="S_bulk.bam", verbose=True)
Pooling(F2_L_bulk, bam_dir="bam/", output="L_bulk.bam", verbose=True)

## IGV表示の準備: ライブラリ
import igv_notebook
from igv_prep import RefTrack, AnnotationTrack, BamTrack
## IGV表示の準備: Reference info
ref = RefTrack({ "fastaPath":"genome/CultivarA.fa", "indexPath":"genome/CultivarA.fa.fai", "id":"CultivarA" })
## IGV表示の準備: Reads info
bam_common_info = { "displayMode":"SQUISHED", "viewAsPairs":True, "height":200 }
S = BamTrack({ **{"name":"S-bulk", "path":"S_bulk.bam", "indexPath":"S_bulk.bam.bai"}, **bam_common_info })
L = BamTrack({ **{"name":"L-bulk", "path":"L_bulk.bam", "indexPath":"L_bulk.bam.bai"}, **bam_common_info })
## IGV表示
igv_notebook.init()
b = igv_notebook.Browser(ref)
b.load_track(S)
b.load_track(L)

### バルクのアリルの割合を読み取る

　IGVで、S-bulk と L-bulk の **55番目** と **120番目** の塩基を見てみましょう。各トラックの一番上のカバレッジ（灰色の棒グラフ）の、その位置をクリックすると、塩基ごとのリードの本数と割合が表示されます。

　55番目では、S-bulk の品種Bの割合はほぼ0、L-bulk はほぼ1になったはずです。では、120番目はどうだったでしょうか？ 人によって選んだ個体が違うので、値もばらばらだと思いますが、基本的に55番目では割合は0.5から大きくずれ、120番目では0.5に近い値となるはずです。

### 確認ポイント
- 原因遺伝子と関係のない場所でも、バルクのアリルの割合が0.5から大きくずれることがあるのはなぜでしょうか？
- バルクに入れる個体数を4個体から20個体に増やすと、120番目の割合のばらつきはどうなるか？ 余裕があれば、`F2_S_bulk` に20個体程度を入れて試してみてください。

　今回、指定した形質の個体を集めてバルクシーケンシングをしました。

<img src="https://github.com/slt666666/FAO_lecture/blob/main/FAO_2024/Mutmap_6.png?raw=true" alt="title" height="300px">

　今回はShort, Longの形質の個体を1個体ずつシーケンスせずとも、まとめて1度ずつシーケンスするだけで、55番目のSNPが原因の変異であると検出できました。

　つまり、バルクシーケンシングの利点は、なんらかの形質に関わる遺伝子のマッピングをおこなう際、分離集団（例えば、F<small>2</small>個体集団）の各個体を個別にシーケンシングするよりも費用を抑えられます。

例）シーケンシング費用が1サンプル10,000円の場合:  
- F<small>2</small>集団100個体の個別シーケンシング: 1,000,000円
- F<small>2</small>集団の二つの形質のバルクシーケンシング: 20,000円


## 2-2. 実例: イネのバルクシーケンシング解析

　では最後にイネのバルクシーケンシング解析の実例を見てみましょう。

　イネいもち病にかかりにくい品種（抵抗性品種）とかかりやすい品種（罹病性品種）の交配からできた分離集団で、抵抗性個体グループと罹病性個体グループをバルクシーケンシングし、罹病性品種のゲノム配列にアライメントしたデータです。

- 参照配列: 罹病性品種のゲノム配列
- バルクシーケンシングリード:
  - 抵抗性個体のバルクシーケンシングリード
  - 罹病性個体のバルクシーケンシングリード

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/qtlseq_takagi_etal_2013_2.png?raw=true" alt="Rice QTL-seq example (Takagi et al., 2013)" height="250px" align="middle">





　次の2つのコードセルを実行すると、イネのアライメント結果（イネ6番染色体の15Mbのみ）を見れます。アライメント結果から、形質と連関がありそうなSNP箇所（または、ゲノム領域）を探してみてください。




In [ ]:
## Colab上にデータファイルのダウンロードする
!pip install -q -U igv-notebook==0.3.1
!wget -q -O igv_prep.py    https://github.com/CropEvol/lecture/raw/master/textbook_2025/scripts/igv_prep.py
!wget -q -O rice_chr6_1_15Mb.fa    https://raw.githubusercontent.com/qqep685d/sample_dataset/main/qtlseq/rice_chr6_1_15Mb.fa
!wget -q -O rice_chr6_1_15Mb.fa.fai https://raw.githubusercontent.com/qqep685d/sample_dataset/main/qtlseq/rice_chr6_1_15Mb.fa.fai
!wget -q -O rice_R_bulk.bam    https://raw.githubusercontent.com/qqep685d/sample_dataset/main/qtlseq/R_bulk.sample.bam
!wget -q -O rice_R_bulk.bam.bai https://raw.githubusercontent.com/qqep685d/sample_dataset/main/qtlseq/R_bulk.sample.bam.bai
!wget -q -O rice_S_bulk.bam    https://raw.githubusercontent.com/qqep685d/sample_dataset/main/qtlseq/S_bulk.sample.bam
!wget -q -O rice_S_bulk.bam.bai https://raw.githubusercontent.com/qqep685d/sample_dataset/main/qtlseq/S_bulk.sample.bam.bai

In [ ]:
import igv_notebook
from igv_prep import RefTrack, AnnotationTrack, BamTrack
igv_notebook.init()
bam_common_info = { "displayMode":"SQUISHED", "alleleFreqThreshold":0.4, "height":200 }

## 準備: Reference info
ref    = RefTrack({ "fastaPath":"rice_chr6_1_15Mb.fa", "indexPath":"rice_chr6_1_15Mb.fa.fai", "id":"Rice" }, locus="chr6:1-10,000")
## 準備: Reads info
r_bulk = BamTrack({ **{"name":"R_bulk", "path":"rice_R_bulk.bam", "indexPath":"rice_R_bulk.bam.bai"}, **bam_common_info })
s_bulk = BamTrack({ **{"name":"S_bulk", "path":"rice_S_bulk.bam", "indexPath":"rice_S_bulk.bam.bai"}, **bam_common_info })
## IGV表示
igv_notebook.init()
b = igv_notebook.Browser(ref)
b.load_track(r_bulk)
b.load_track(s_bulk)

　イネの染色体は12本あります。今回表示したデータは、イネ6番染色体の一部領域のみ（chr6:1-15,000,000）です。この領域に形質と連関するSNP箇所が含まれていますが、このアライメント結果では多量のSNPがあり、目視でそのSNP箇所を探すのは大変です。

　次回の実習で、SNP情報をPythonプログラミングで解析し、原因遺伝子があるゲノム領域の探索をしていきましょう。

# まとめ

　今回、リードアライメントとSNP検出、バルクシーケンシングについて勉強しました。
- リードアライメント
  - 似た配列の場所を参照配列から探し、参照配列にリードを並べる
- SNP検出
  - 参照配列とリード配列を比較し、信頼性のある塩基の違い（SNP）を探す
- ジェノタイピングと遺伝子マッピング
  - 原因遺伝子の近くのSNPほど形質との連関が強く、離れるほど組換えによって連関が弱くなる
  - リードの本数が少ないと、偶然の偏りによって遺伝子型の判定を誤ることがある
- バルクシーケンシング
  - 複数個体のDNAをプーリングし、そのバルクDNAをシーケンシングする
  - とくに、遺伝子マッピングでは、形質ごとにバルクDNAを用意し、バルクシーケンシングをおこなう

  


<div align="right"><a href="https://github.com/CropEvol/lecture#section2">実習表ページに戻る</a></div>

#### ソフトウェアのインストールがうまくいかない場合
　次のコードセルを実行して、別の方法でのインストールを試してみてください。

In [ ]:
### ソフトウェアのインストール
## igv viewer
!pip install -q igv-notebook
!wget -q -O igv_prep.py https://github.com/CropEvol/lecture/raw/master/textbook_2025/scripts/igv_prep.py
## conda-env
!pip install -q condacolab
import condacolab
condacolab.install_miniforge()
## softwares
!conda install -q -y bioconda::bwa
!apt-get -qq install -y samtools > /dev/null 2>&1